# HPO SolarSystemLander

Study Series 2A (8D) and 2B (11D) from `hpo/design3.md`.

## Setup

In [ ]:
# Colab setup
!git clone https://github.com/miwehle/rl_lab.git
%cd rl_lab
!pip install -r hpo/requirements.txt

from pathlib import Path
import sys
from google.colab import drive

sys.path.insert(0, "dqn/src")
sys.path.insert(0, "hpo/src")

from hpo.drive_backup import backup_to_drive, restore_from_drive
from hpo.lunar_lander.logging import configure_file_logging

drive.mount("/content/drive")
DRIVE_STUDY_DIR = Path("/content/drive/MyDrive/rl_lab/hpo")
LOCAL_STUDY_DIR = Path("/content/rl_lab/hpo/runs")
DRIVE_STUDY_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_STUDY_DIR.mkdir(parents=True, exist_ok=True)

## Configuration

In [ ]:
from dataclasses import replace
import torch

from dqn.vector_training import VectorTrainingConfig
from hpo.evaluation.scoring import ScoringConfig
from hpo.objective import TrialConfig
from hpo.solar_system_lander.environment import SolarSystemLanderEnvironmentFactory
from hpo.study import StudyRunner, neighbors

OBSERVATION_MODE = "8d_series3"  # Series 2A; use "11d" for Series 2B
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
TRIAL_CFG = TrialConfig(num_envs=20, device=device)
SCORING_CFG = ScoringConfig(quality_weight=0.7, eval_episodes=10)
ENVIRONMENT_FACTORY = SolarSystemLanderEnvironmentFactory(OBSERVATION_MODE)
RUN_NAME = f"solar_system_lander_{OBSERVATION_MODE}"
STORAGE_PATH = LOCAL_STUDY_DIR / f"{RUN_NAME}.db"
DRIVE_STORAGE_PATH = DRIVE_STUDY_DIR / f"{RUN_NAME}.db"
LOG_PATH = LOCAL_STUDY_DIR / f"{RUN_NAME}.log"
DRIVE_LOG_PATH = DRIVE_STUDY_DIR / f"{RUN_NAME}.log"
restore_from_drive(DRIVE_STORAGE_PATH, STORAGE_PATH)
restore_from_drive(DRIVE_LOG_PATH, LOG_PATH)
configure_file_logging(LOCAL_STUDY_DIR, LOG_PATH.name)

def backup_study_to_drive():
    backup_to_drive(
        local_database=STORAGE_PATH,
        drive_database=DRIVE_STORAGE_PATH,
        local_log=LOG_PATH,
        drive_log=DRIVE_LOG_PATH,
    )

device, STORAGE_PATH, DRIVE_STORAGE_PATH

## Search spaces

In [ ]:
BATCH_SIZES = [256, 512, 1024]
LEARNING_STARTS = [1_000, 2_500, 5_000]
OPTIMIZE_EVERY = [2, 4, 8]
REPLAY_CAPACITIES = [100_000, 200_000, 400_000]
EPISODE_COUNTS = [500, 1_000, 1_500]

BASELINE_PARAMS = {
    "learning_rate": 0.001606,
    "batch_size": 1024,
    "eps_end": 0.0197,
    "eps_decay": 8_446,
    "gamma": 0.99,
    "tau": 0.005,
    "learning_starts": 2_500,
    "optimize_every": 4,
    "replay_memory_capacity": 200_000,
    "num_episodes": 600,
}

def vector_config(params):
    return VectorTrainingConfig(
        num_episodes=params["num_episodes"],
        batch_size=params["batch_size"],
        gamma=0.99,
        eps_start=1.0,
        eps_end=params["eps_end"],
        eps_decay=params["eps_decay"],
        tau=0.005,
        learning_rate=params["learning_rate"],
        learning_starts=params["learning_starts"],
        optimize_every=params["optimize_every"],
    )

class SearchSpace0:
    def training_config(self, trial, params):
        return vector_config(params)

    def replay_memory_capacity(self, trial, params):
        return params["replay_memory_capacity"]

class SearchSpace1:
    def training_config(self, trial, params):
        return vector_config(params | {
            "learning_rate": trial.suggest_float("learning_rate", 5e-4, 3e-3, log=True),
            "batch_size": trial.suggest_categorical("batch_size", BATCH_SIZES),
            "learning_starts": trial.suggest_categorical("learning_starts", LEARNING_STARTS),
            "optimize_every": trial.suggest_categorical("optimize_every", OPTIMIZE_EVERY),
            "num_episodes": trial.suggest_categorical("num_episodes", EPISODE_COUNTS),
        })

    def replay_memory_capacity(self, trial, params):
        return params["replay_memory_capacity"]

class SearchSpace2:
    def training_config(self, trial, params):
        return vector_config(params | {
            "eps_end": trial.suggest_float("eps_end", 0.01, 0.10),
            "eps_decay": trial.suggest_int("eps_decay", 5_000, 100_000, log=True),
        })

    def replay_memory_capacity(self, trial, params):
        return params["replay_memory_capacity"]

class SearchSpace3:
    def training_config(self, trial, params):
        return vector_config(params)

    def replay_memory_capacity(self, trial, params):
        return trial.suggest_categorical("replay_memory_capacity", REPLAY_CAPACITIES)

class SearchSpace4:
    def training_config(self, trial, params):
        return vector_config(params | {
            "learning_rate": trial.suggest_float("learning_rate", params["learning_rate"] / 2, params["learning_rate"] * 2, log=True),
            "batch_size": trial.suggest_categorical("batch_size", neighbors(params["batch_size"], BATCH_SIZES)),
            "eps_end": trial.suggest_float("eps_end", max(0.01, params["eps_end"] - 0.02), min(0.10, params["eps_end"] + 0.02)),
            "eps_decay": trial.suggest_int("eps_decay", max(1, params["eps_decay"] // 2), params["eps_decay"] * 2, log=True),
            "learning_starts": trial.suggest_categorical("learning_starts", neighbors(params["learning_starts"], LEARNING_STARTS)),
            "optimize_every": trial.suggest_categorical("optimize_every", neighbors(params["optimize_every"], OPTIMIZE_EVERY)),
            "num_episodes": trial.suggest_categorical("num_episodes", neighbors(params["num_episodes"], EPISODE_COUNTS)),
        })

    def replay_memory_capacity(self, trial, params):
        return params["replay_memory_capacity"]


## Optimization

In [ ]:
runner = StudyRunner(
    storage_path=lambda _name: STORAGE_PATH,
    environment_factory=ENVIRONMENT_FACTORY,
    trial_cfg=TRIAL_CFG,
    incumbent_params=BASELINE_PARAMS,
    study_attrs=ENVIRONMENT_FACTORY.metadata(),
    extra_seeds=(1001,),
    sync_fn=backup_study_to_drive,
)

runner.run(
    "s0_qe_baseline", SearchSpace0(), 3, SCORING_CFG, robust=False,
)
study0 = runner.studies[-1]
study0.set_user_attr("baseline_params", BASELINE_PARAMS)
backup_study_to_drive()
scoring_cfg = replace(
    SCORING_CFG,
    baseline_env_steps=study0.user_attrs["baseline_env_steps"],
    baseline_processed_samples=study0.user_attrs["baseline_processed_samples"],
)
runner.run("s1_qe_update_economy", SearchSpace1(), 40, scoring_cfg)
runner.run("s2_qe_exploration", SearchSpace2(), 25, scoring_cfg)
runner.run("s3_qe_replay_capacity", SearchSpace3(), 10, scoring_cfg)
runner.run("s4_qe_joint_finetune", SearchSpace4(), 25, scoring_cfg)


## Reload studies after a runtime reset

In [ ]:
# Run only after the runtime environment has been reset.
import optuna

def load_study(name):
    return optuna.load_study(study_name=name, storage=f"sqlite:///{STORAGE_PATH}")

study0 = load_study("s0_qe_baseline")
study1 = load_study("s1_qe_update_economy")
study2 = load_study("s2_qe_exploration")
study3 = load_study("s3_qe_replay_capacity")
study4 = load_study("s4_qe_joint_finetune")

## Analysis

In [ ]:
import pandas as pd

studies = [study0, study1, study2, study3, study4]
labels = ["S0", "S1", "S2", "S3", "S4"]

def selected_trial(study):
    params = study.user_attrs.get("robust_best_params")
    candidates = [trial for trial in study.trials if trial.params == params]
    return max(candidates, key=lambda trial: trial.value) if candidates else study.best_trial

rows = []
for label, study in zip(labels, studies):
    trial = selected_trial(study)
    rows.append({
        "study": label,
        "objective_score": study.user_attrs.get("robust_best_objective_score", trial.value),
        "gym_score": study.user_attrs.get("robust_best_gym_score", trial.user_attrs["gym_score"]),
        "training_effort": study.user_attrs.get("robust_best_training_effort", trial.user_attrs["training_effort"]),
        **trial.user_attrs.get("gym_scores", {}),
    })

display(pd.DataFrame(rows).set_index("study").style.format("{:.1f}"))